# GP_KDB — module tests

Read-only tests for **MongoDB** access via the `KDB` singleton.

- Databases: `kdb` (calendar), `tdx` (quotes/stocks), `site` (chart settings)
- Set `TEST_IDE` to any stock code you want to inspect
- Requires MongoDB running (default URI from `GP_KDB_MONGO_URI` or `GP_TDX_MONGO_URI`)

In [1]:
import sys
sys.path.insert(0, r'C:/Project/modules')

from GP_KDB import KDB
from GP_KDB.config import DEFAULT_CONFIG

TEST_IDE = 'sh600519'   # change to any IDE, e.g. sz000001, sh689009

print('Mongo URI:', DEFAULT_CONFIG.mongo_uri)
print('Mongo host:', DEFAULT_CONFIG.mongo_ip, ':', DEFAULT_CONFIG.mongo_port)

Mongo URI: mongodb://127.0.0.1:27017
Mongo host: localhost : 27017


## 1. Connect to MongoDB

Calls `KDB().setting()` — creates indexes and loads the trading calendar.

In [3]:
kdb = KDB().setting()

print('Connected:', kdb.is_configured)
print('Trading days:', len(kdb.DTs))
print('First date:', kdb.DTs[0] if kdb.DTs else None)
print('Last date:', kdb.DTs[-1] if kdb.DTs else None)
print('STOCK count:', kdb.col_TDX_STOCK.estimated_document_count())
print('QUATE count:', kdb.col_TDX_QUATE.estimated_document_count())

Connected: True
Trading days: 15000
First date: 1990-12-19
Last date: 2032-01-12
STOCK count: 5357
QUATE count: 28371031


## 2. Stock info (`StockInfo`, `SecurityType`, `stocksIDEs`)

In [4]:
info = kdb.StockInfo(TEST_IDE)
print('IDE:', TEST_IDE)
print('Security type:', kdb.SecurityType(TEST_IDE))
print('Info keys:', list(info.keys()) if info else 'not found')
info

IDE: sh600519
Security type: stock
Info keys: ['IDE', 'ABB', 'DQ', 'HY1', 'HY2', 'HY3', 'ID6', 'IDS', 'index', 'BK_GN', 'BK_FG', 'oldIDE']


{'IDE': 'sh600519',
 'ABB': 'GZMT',
 'DQ': ['贵州', 'sh880229', '29'],
 'HY1': ['食品饮料', 'sh881129', 'X21'],
 'HY2': ['酿酒', 'sh881130', 'X2102'],
 'HY3': ['白酒', 'sh881135', 'X210205'],
 'ID6': '600519',
 'IDS': '贵州茅台',
 'index': [['沪深300', 'sh000300'],
  ['沪深300', 'sz399300'],
  ['上证50', 'sh000016'],
  ['中证100', 'sh000903'],
  ['中证100', 'sz399903'],
  ['上证180', 'sh000010'],
  ['央视50', 'sz399550'],
  ['腾讯济安', 'sh000847'],
  ['投资时钟', 'sz399391']],
 'BK_GN': [['白酒概念', 'sh880564']],
 'BK_FG': [['融资融券', ''],
  ['高分红股', 'sh880526'],
  ['行业龙头', 'sh880847'],
  ['大盘股', 'sh880821'],
  ['高市净率', 'sh880827'],
  ['北上重仓', 'sh880721'],
  ['基金重仓', 'sh880801'],
  ['证金汇金', ''],
  ['绩优股', 'sh880835'],
  ['持续增长', 'sh880895'],
  ['MSCI成份', 'sh880883'],
  ['百元股', 'sh880878'],
  ['昨成交20', 'sh880789'],
  ['陆股通重', ''],
  ['非周期股', 'sh880680']],
 'oldIDE': '6005191'}

In [5]:
all_ides = kdb.stocksIDEs()
print(f'Total stocks: {len(all_ides):,}')
print('Sample:', all_ides[:5])

Total stocks: 5,357
Sample: ['bj430017', 'bj430047', 'bj430090', 'bj430139', 'bj430198']


## 3. Daily quotes (`DK_STOCK`)

Raw OHLCV from `tdx.QUATE`.

In [6]:
dk = kdb.DK_STOCK(TEST_IDE)
print('Shape:', dk.shape)
dk.tail(10)

Shape: (5946, 9)


,IDE,DT,O,C,H,L,V,A,DTnum
5936,sh600519,2026-06-10,1252.08,1275.88,1282.00,1250.21,39244.14,4.991687e+09,12957
5937,sh600519,2026-06-11,1272.12,1279.00,1282.88,1266.91,25351.98,3.230008e+09,12958
5938,sh600519,2026-06-12,1271.18,1291.91,1295.00,1265.01,50494.78,6.477910e+09,12959
5939,sh600519,2026-06-15,1292.70,1271.10,1292.70,1270.10,41585.56,5.303656e+09,12962
5940,sh600519,2026-06-16,1267.01,1255.67,1267.88,1255.00,34969.74,4.399958e+09,12963
5941,sh600519,2026-06-17,1258.00,1240.00,1259.77,1238.56,44803.30,5.581256e+09,12964
5942,sh600519,2026-06-18,1235.00,1215.00,1238.87,1211.22,57471.73,7.016714e+09,12965
5943,sh600519,2026-06-22,1214.31,1241.41,1252.80,1205.00,58251.31,7.163236e+09,12969
5944,sh600519,2026-06-23,1239.00,1222.45,1264.00,1217.00,58004.05,7.179746e+09,12970
5945,sh600519,2026-06-24,1222.65,1207.68,1241.87,1207.51,45335.28,5.516574e+09,12971


## 4. Adjusted quotes (`FDK_STOCK`)

Forward-adjusted prices using `STOCK_PKVZGB` (splits / total shares).

In [7]:
fh = kdb.FH(TEST_IDE)
print('FH shape:', fh.shape)
fh.tail(5)

FH shape: (43, 4)


,DT,IDE,PKV,ZGB
38,2025-06-26,sh600519,5.116151,1.256198e+09
39,2025-09-01,sh600519,5.116151,1.252270e+09
40,2025-12-19,sh600519,5.116151,1.252270e+09
41,2026-05-28,sh600519,5.116151,1.250082e+09
42,2026-06-26,sh600519,5.116151,1.250082e+09


In [8]:
fdk = kdb.FDK_STOCK(TEST_IDE)
print('FDK shape:', fdk.shape)
fdk[['DT', 'O', 'H', 'L', 'C', 'V', 'T', 'MP']].tail(10)

FDK shape: (5946, 13)


,DT,O,H,L,C,V,T,MP
5936,2026-06-10,880.871599,883.244906,880.721390,882.763996,7670.637495,0.313933,2823.812275
5937,2026-06-11,882.467389,883.313868,882.054947,883.009453,4955.283728,0.202803,2824.057733
5938,2026-06-12,882.393100,884.258876,881.904114,884.018789,9869.681252,0.403932,2825.067068
5939,2026-06-15,884.080225,884.080225,882.307679,882.386775,8128.290130,0.332663,2823.435054
5940,2026-06-16,882.062879,882.131864,881.105703,881.159341,6835.165680,0.279740,2822.207620
5941,2026-06-17,881.345653,881.486955,879.780505,879.897281,8757.227777,0.358403,2820.945560
5942,2026-06-18,879.491223,879.805656,877.537233,877.850384,11233.391968,0.459744,2818.898663
5943,2026-06-22,877.793294,880.929374,877.019808,880.011494,11385.768236,0.465980,2821.059773
5944,2026-06-23,879.816201,881.823842,878.015679,878.464732,11337.438935,0.464002,2819.513011
5945,2026-06-24,878.481173,880.048726,877.228929,877.243076,8861.208288,0.362658,2818.291356


## 5. Financial statements (`CW_STOCK`)

Quarterly financial data from `tdx.CW`.

In [9]:
cw = kdb.CW_STOCK(TEST_IDE)
print('CW shape:', cw.shape)
cw.tail(5) if not cw.empty else 'No CW data for this IDE'

CW shape: (106, 585)


,CODE,col1,col10,col100,col101,col102,col103,col104,col105,col106,...,col90,col91,col92,col93,col94,col95,col96,col97,col98,col99
REPORTDATE,,,,,,,,,,,,,,,,,,,,,
2025-03-31,600519,21.38,2.127459e+09,1.689179e+09,5.058425e+10,2.663148e+09,7.662339e+09,2.216816e+10,1.126582e+09,4.177505e+10,...,0.0,0.0,3.703145e+10,9.256818e+09,0.0,2.777464e+10,2.684747e+10,9.271618e+08,5.614406e+10,0.0
2025-06-30,600519,36.18,2.815781e+09,3.959174e+09,9.800253e+10,6.076043e+09,9.965785e+09,4.166989e+10,4.071847e+09,8.488347e+10,...,0.0,0.0,6.277800e+10,1.579132e+10,0.0,4.698668e+10,4.540296e+10,1.583719e+09,9.508662e+10,0.0
2025-09-30,600519,51.53,5.209530e+09,4.429718e+09,1.367335e+11,8.595278e+09,1.239348e+10,5.910871e+10,7.567357e+09,9.853672e+10,...,0.0,0.0,8.940344e+10,2.250464e+10,0.0,6.689880e+10,6.462675e+10,2.272058e+09,1.387917e+11,0.0
2025-12-31,600519,65.66,0.000000e+00,4.712295e+09,1.861288e+11,1.145014e+10,1.547346e+10,7.743060e+10,9.012158e+09,1.246066e+11,...,0.0,0.0,1.147553e+11,2.944494e+10,0.0,8.531032e+10,8.232006e+10,2.990258e+09,1.839904e+11,0.0
2026-03-31,600519,21.76,0.000000e+00,4.154339e+08,5.618097e+10,3.211262e+09,7.351815e+09,2.403097e+10,1.463836e+09,2.927107e+10,...,0.0,0.0,3.754325e+10,9.389418e+09,0.0,2.815383e+10,2.724251e+10,9.113186e+08,5.639259e+10,0.0


## 6. Industry blocks (`HYchildren`, `stocks_in_HY`)

In [10]:
hy_top = kdb.HYchildren('全部行业')
print('Top-level industries:', len(hy_top['children']))
hy_top['children'][:5]

Top-level industries: 30


[['煤炭', 'sh881001', 'X10'],
 ['石油', 'sh881006', 'X11'],
 ['化工', 'sh881015', 'X12'],
 ['钢铁', 'sh881061', 'X13'],
 ['有色', 'sh881070', 'X14']]

In [11]:
if info and 'HY1' in info:
    hy_ids = info['HY1'][0]
    stocks = kdb.stocks_in_HY(hy_ids)
    print(f'Industry {hy_ids}: {len(stocks)} stocks')
    stocks[:10]
else:
    print('No HY1 field on this stock')

Industry 食品饮料: 141 stocks


## 7. Block mapping (`StocksByBK`)

Expands each stock into its block memberships. This can be large — showing a filtered sample.

In [12]:
bk = kdb.StocksByBK()
print('Total IDE-BK rows:', len(bk))
bk[bk['IDE'] == TEST_IDE]

Total IDE-BK rows: 57110


,IDE,BK
5830,sh600519,sh
5831,sh600519,sh880229
5835,sh600519,sh880564
5832,sh600519,sh881129
5833,sh600519,sh881130
5834,sh600519,sh881135


## 8. Batch query (`getDKs_in_Duration`)

Load last N trading days for multiple stocks (aggregation pipeline).

In [30]:
# LAST_DT = kdb.DTs[12000]
SAMPLE_IDES = all_ides[1000:1005]
SAMPLE_IDES
batch = kdb.getDKs_in_Duration(ides=SAMPLE_IDES, dt='2025-10-14', duration=20)
print('Batch shape:', batch.shape)
batch.head(10)

Batch shape: (100, 11)


,IDE,DT,O,C,H,L,V,A,ZGB,PKV,T
30,sh600979,2025-09-09,401.107390,400.080831,401.107390,399.667254,246060.44,120091728.0,1.261657e+09,10.999422,0.000195
31,sh600979,2025-09-10,400.286984,399.874255,400.286984,399.459825,187950.01,91312040.0,1.261657e+09,10.999422,0.000149
32,sh600979,2025-09-11,399.667254,400.698024,400.698024,398.625802,284989.11,138276304.0,1.261657e+09,10.999422,0.000226
33,sh600979,2025-09-12,401.107390,400.492714,401.311449,400.286984,277045.80,135979776.0,1.261657e+09,10.999422,0.000220
34,sh600979,2025-09-15,400.286984,399.667254,400.902915,399.667254,210780.73,102648912.0,1.261657e+09,10.999422,0.000167
35,sh600979,2025-09-16,399.874255,401.718329,401.718329,399.459825,377429.12,185136000.0,1.261657e+09,10.999422,0.000299
36,sh600979,2025-09-17,402.728379,401.718329,404.323638,401.718329,370816.91,184955792.0,1.261657e+09,10.999422,0.000294
37,sh600979,2025-09-18,401.921152,400.080831,401.921152,399.459825,379002.38,184878560.0,1.261657e+09,10.999422,0.000300
38,sh600979,2025-09-19,399.667254,398.834958,399.874255,398.416211,226209.84,108853272.0,1.261657e+09,10.999422,0.000179
39,sh600979,2025-09-22,399.459825,402.929177,404.323638,399.251968,572291.87,284519680.0,1.261657e+09,10.999422,0.000454


## 9. Site database (`ChartDomain`)

Read chart layout settings from `site.chartdomain`.

In [31]:
chart_names = [x['chartname'] for x in kdb.col_chartdomain.find({}, {'chartname': 1, '_id': 0}).limit(5)]
print('Sample chart names:', chart_names)

if chart_names:
    kdb.ChartDomain(chart_names[0])
else:
    print('No chart domains in site DB yet')

Sample chart names: ['HY_光伏辅材_001', 'HY_电力设备_001', 'HY_光伏设备_001', 'sh_001', 'sh601865_001']


## 10. Cleanup

In [32]:
kdb.close()
print('Connection closed')

Connection closed
